In [ ]:
# 1. Install Unsloth and TRL for optimized RL training
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install openenv-core pydantic

# 2. Clone your repo (use your actual GitHub link)
!git clone https://github.com/Atul-Kumar29/Leakguarddemo
import sys
import os
sys.path.append('/content/LeakGuardAI')

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-s01m9ars/unsloth_fa048fe440794c84a5be0369f9a3c2d9
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-s01m9ars/unsloth_fa048fe440794c84a5be0369f9a3c2d9
  Resolved https://github.com/unslothai/unsloth.git to commit 5c473fab80e079bb525345b86cb71afd409262c3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.8 MB/s eta 0:00:00

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

# Fetch the token from the Secrets menu
token = userdata.get('HF_TOKEN')

# Log in
login(token=token)

# Also set it as an environment variable for Git/Transformers
os.environ["HF_TOKEN"] = token

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    use_gradient_checkpointing = "unsloth",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
import re
import json
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import sys
sys.path.append('/content/Leakguarddemo')
from server.environment import LeakGuardEnvironment

# Initialize local environment
env = LeakGuardEnvironment()

def reward_logic(completions, **kwargs):
    """Executes actions and returns rewards from your logic."""
    rewards = []
    for content in completions:
        text = content[0]['content'] if isinstance(content, list) else content
        try:
            # Extract JSON and step environment
            match = re.search(r'\{.*\}', text, re.DOTALL)
            action = json.loads(match.group(0))
            _, reward, done, _ = env.step(action)
            if done: env.reset()
            rewards.append(float(reward))
        except:
            rewards.append(-1.0) # Penalty for invalid JSON or IDs
    return rewards

# Prepare 100 training prompts
prompts = [{"prompt": [{"role": "system", "content": "You are a LeakGuard auditor. Output JSON: {'invoice_id': int, 'decision': 'APPROVE|REJECT|NEGOTIATE'}"},
           {"role": "user", "content": f"State:\n{env.reset()}"}]} for _ in range(100)]
dataset = Dataset.from_dict({"prompt": [p["prompt"] for p in prompts]})

# Configure GRPO Trainer
training_args = GRPOConfig(
    output_dir = "leakguard_rl",
    learning_rate = 5e-6,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    max_steps = 60,
    logging_steps = 1,
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = [reward_logic],
    args = training_args,
    train_dataset = dataset,
)

trainer.train()

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151654}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been s

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_logic / mean,rewards / reward_logic / std
1,0.006877,0.033844,0.095442,110.781250,25.000000,213.000000,0.000000,110.781250,25.000000,213.000000,0.000015,0.033844,0.173554
2,-0.086432,0.064147,0.171498,60.187500,24.000000,130.000000,0.000000,60.187500,24.000000,130.000000,0.000000,0.064147,0.227938
3,0.066864,0.034466,0.097201,56.468750,24.000000,107.000000,0.000000,56.468750,24.000000,107.000000,0.000008,0.034466,0.177054
4,0.113048,0.065088,0.174167,101.031250,24.000000,230.000000,0.000000,101.031250,24.000000,230.000000,0.000015,0.065087,0.243273
5,-0.043515,0.067176,0.180081,89.875000,23.000000,191.000000,0.000000,89.875000,23.000000,191.000000,0.000013,0.067176,0.239001
6,0.003833,0.034466,0.087547,71.531250,24.000000,137.000000,0.000000,71.531250,24.000000,137.000000,0.000010,0.034466,0.177054
7,-0.097746,0.068831,0.184747,79.031250,20.000000,203.000000,0.000000,79.031250,20.000000,203.000000,0.000010,0.068831,0.245475
8,0.012687,0.034466,0.097201,51.250000,23.000000,117.000000,0.000000,51.250000,23.000000,117.000000,0.000012,0.034466,0.177054
9,-0.023479,0.061025,0.172322,82.968750,23.000000,167.000000,0.000000,82.968750,23.000000,167.000000,0.000013,0.061025,0.228115
10,0.047789,-0.031163,0.204110,87.000000,20.000000,180.000000,0.000000,87.000000,20.000000,180.000000,0.000026,-0.031163,0.309460


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

TrainOutput(global_step=60, training_loss=0.008707768435124308, metrics={'train_runtime': 13684.1759, 'train_samples_per_second': 0.14, 'train_steps_per_second': 0.004, 'total_flos': 0.0, 'train_loss': 0.008707768435124308})

In [ ]:
from google.colab import userdata

# 1. Refresh the token variable from Colab Secrets
# This ensures it uses the 'Classic Write Token' you just updated
final_token = userdata.get('HF_TOKEN')

# 2. Save the trained adapters locally as a backup
print("Saving model locally...")
model.save_pretrained("leakguard_final_model")
tokenizer.save_pretrained("leakguard_final_model")

# 3. Push to Hugging Face (Step A: LoRA Adapters)
# This will automatically create the 'LeakGuard-RL-Auditor' repo if it doesn't exist
print("Pushing to Hugging Face Hub... this may take a few minutes.")
model.push_to_hub(
    "AtulK29/LeakGuard-RL-Auditor",
    token = final_token
)
tokenizer.push_to_hub(
    "AtulK29/LeakGuard-RL-Auditor",
    token = final_token
)

print("---")
print("SUCCESS: Your audit brain is now live at https://huggingface.co/AtulK29/LeakGuard-RL-Auditor")

Saving model locally...


Unsloth: Restored added_tokens_decoder metadata in leakguard_final_model/tokenizer_config.json.


Pushing to Hugging Face Hub... this may take a few minutes.


README.md:   0%|          | 0.00/577 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpsjvdse27/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpsjvdse27/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

---
SUCCESS: Your audit brain is now live at https://huggingface.co/AtulK29/LeakGuard-RL-Auditor
